# Task 2 – Lesk algorithm on the MSH-WSD dataset

> *Make a random selection of 10 acronyms and 10 regular terms. Use Lesk algorithm to perform the word sense
> disambiguation of each of these 10 acronyms and 10 regular terms and use the ground truth to compute the
> average accuracy on regular terms and average accuracy on acronyms.*

MSH-WSD contains medical abstracts where an ambiguous word is marked, together with its correct meaning. We let Lesk to guess the meaning of the word in each abstract and then we grade its answer and we compare it against the solutions.

Lesk and MSH-WSD do not speak the same language:
- MSH-WSD writes the correct meaning as a **UMLS concept** (CUI), e.g. `C0009443`;
- Lesk answers with a **WordNet synset**, e.g. `cold.n.01`.


**Note on the Lesk implementation** \
Although the NLTK documentation describes `nltk.wsd.lesk` as "an implementation of the original Lesk
algorithm (1986)", inspecting the source code shows that it implements the **simplified Lesk** algorithm
(Kilgarriff & Rosenzweig, 2000): each sense is scored by the overlap between its WordNet gloss and the
**context words**, whereas the original algorithm (Lesk, 1986) compares the gloss of each sense with the
**glosses of the other words** in the context.


In [1]:
import sys
sys.path.append("..")
from wsd_utils import (CONCEPT_MAP as concept_map, load_instances,
                       ALL_ACRONYMS as acronyms, ALL_TERMS as terms, wordnet_senses, MAPPING, mean)
import re
import random
import nltk
import pandas as pd
import requests
from getpass import getpass
from nltk.corpus import wordnet as wn
from collections import Counter
from nltk.wsd import lesk
from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)  

STOPWORDS = set(stopwords.words("english"))


**Data preparation**

The dataset has two kinds of files. `benchmark_mesh.txt` has one line per ambiguous word, with the codes (CUIs) of all its meanings, e.g. `AA C0002520 C0001972`. Then there is one file per word (e.g. `AA_pmids_tagged.arff`) with its abstracts: in each abstract the ambiguous word is marked as `<e>word</e>` and the correct meaning is written as a label, `M1` = first CUI of the line, `M2` = second CUI, and so on. We read each abstract into a small record with the clean text (`context`) and the correct answer (`cui`).

The dataset does not say which words are acronyms, so we use a simple rule: a word is an acronym if it has **at least two capital letters** (`AA`, `BPD`) or **contains a digit** (`CCl4`). The rule makes a few mistakes, so we correct them by hand:
- abbreviations with only one capital letter: *Ca* (calcium), *Ala* (alanine), *Fe* (iron), *dC* (deoxycytidine), *pI* (isoelectric point), *Epi*, *Staph*, *Strep*;
- normal words written in capitals: *ERUPTION*, *LABOR*, *NEUROFIBROMATOSIS*, *THYMUS*, *Gamma-Interferon*;
- three words can be both a term and an abbreviation (*EGG*, *STEM*, *Orf*), so we **exclude** them.

We obtain 108 acronyms and 92 terms, close to the numbers reported in the MSH-WSD paper (106 abbreviations, 88 terms and 9 mixed cases). Then we randomly pick 10 words from each group; `random.seed(42)` makes the selection **reproducible**.

All these functions are in `wsd_utils.py`, shared by all our tasks: here we only show what they return.

In [2]:
print("Ambiguous words:", len(concept_map))
print("Acronyms:", len(acronyms), "  Terms:", len(terms))

# Random selection of 10 acronyms and 10 terms (seed 42, reproducible)
random.seed(42)
selected_acronyms = sorted(random.sample(acronyms, 10))
selected_terms    = sorted(random.sample(terms, 10))

print("\nSelected acronyms:", selected_acronyms)
print("Selected terms   :", selected_terms)

Ambiguous words: 203
Acronyms: 108   Terms: 92

Selected acronyms: ['ADP', 'CAD', 'CAM', 'CDA', 'Ca', 'DDD', 'DON', 'SARS', 'Staph', 'TNT']
Selected terms   : ['B-Cell Leukemia', 'Borrelia', 'Cilia', 'Follicles', 'Ganglion', 'Nurse', 'Pneumocystis', 'Respiration', 'Sodium', 'Synapsis']


**WordNet senses and UMLS names**

To build the CUI → WordNet translation we need to see both sides:
- **WordNet**: the senses of the word, i.e. the only answers Lesk can give. Lesk does not know anything about medicine, so if the correct medical meaning is not in WordNet, Lesk can never find it. (`wordnet_senses` in `wsd_utils.py` also handles spaces and plurals, e.g. *Follicles* → *follicle*.)
- **UMLS**: the dataset gives the correct meanings only as codes (e.g. *CAD* → `C0011905`, `C1956346`), so we ask the **UMLS REST API** for the name of each code (*Computer Assisted Diagnosis*, *Coronary Artery Disease*).

Putting the two lists side by side, we can then choose by hand which WordNet sense matches each code.

In [3]:
API_KEY = getpass("Paste your UMLS API key: ")


def cui_name(cui):
    """Name of a UMLS concept (CUI), from the UMLS REST API (None if not found, e.g. retired CUI)."""
    url = f"https://uts-ws.nlm.nih.gov/rest/content/current/CUI/{cui}"
    r = requests.get(url, params={"apiKey": API_KEY}, timeout=20)
    if r.status_code != 200:
        return None
    return r.json()["result"]["name"]


cui_names = {}      # CUI -> UMLS name, filled while we print


pd.set_option("display.max_rows", 200)        # show all the rows of the table
pd.set_option("display.max_colwidth", 100)    # do not cut the definitions too early


def senses_table(words):
    """For each word: the UMLS names of its CUIs (correct answers) and its WordNet senses (Lesk's possible answers)."""
    rows = []
    for word in words:
        for label, cui in concept_map[word].items():
            if cui not in cui_names:
                cui_names[cui] = cui_name(cui)
            rows.append({"word": word, "source": "MSH-WSD (UMLS)", "id": cui, "meaning": cui_names[cui]})
        senses = wordnet_senses(word)
        if not senses:
            rows.append({"word": word, "source": "WordNet", "id": "-", "meaning": "NOT IN WORDNET"})
        for s in senses:
            rows.append({"word": word, "source": "WordNet", "id": s.name(), "meaning": s.definition()})
    return pd.DataFrame(rows).set_index(["word", "source"])


senses_table(selected_acronyms + selected_terms)

id  \
word            source                                                   
ADP             MSH-WSD (UMLS)                                C0001459   
                MSH-WSD (UMLS)                                C0004374   
                WordNet                     adenosine_diphosphate.n.01   
                WordNet                 automatic_data_processing.n.01   
CAD             MSH-WSD (UMLS)                                C0011905   
                MSH-WSD (UMLS)                                C1956346   
                WordNet                                       cad.n.01   
                WordNet                     computer-aided_design.n.01   
CAM             MSH-WSD (UMLS)                                C0007578   
                MSH-WSD (UMLS)                                C0178551   
                WordNet                                       cam.n.01   
                WordNet                                       cam.n.02   
CDA             MSH-WSD (UMLS)                                C0092801   
                MSH-WSD (UMLS)                                C0002876   
                WordNet                                              -   
Ca              MSH-WSD (UMLS)                                C0006675   
                MSH-WSD (UMLS)                                C0006754   
                MSH-WSD (UMLS)                                C0019564   
                MSH-WSD (UMLS)                                C0006823   
                WordNet                                   calcium.n.01   
                WordNet                                california.n.01   
DDD             MSH-WSD (UMLS)                                C0026256   
                MSH-WSD (UMLS)                                C0011037   
                WordNet                                              -   
DON             MSH-WSD (UMLS)                                C0028652   
                MSH-WSD (UMLS)                                C0012020   
                WordNet                                       don.n.01   
                WordNet                                 preceptor.n.01   
                WordNet                                       don.n.03   
                WordNet                                       don.n.04   
                WordNet                                       don.n.05   
                WordNet                                       don.n.06   
                WordNet                                      wear.v.09   
SARS            MSH-WSD (UMLS)                                C1175175   
                MSH-WSD (UMLS)                                C1175743   
                WordNet         severe_acute_respiratory_syndrome.n.01   
Staph           MSH-WSD (UMLS)                                C0038160   
                MSH-WSD (UMLS)                                C0038170   
                WordNet                            staphylococcus.n.01   
TNT             MSH-WSD (UMLS)                                C0077404   
                MSH-WSD (UMLS)                                C0041070   
                WordNet                                       tnt.n.01   
B-Cell Leukemia MSH-WSD (UMLS)                                C2004493   
                MSH-WSD (UMLS)                                C0023434   
                WordNet                                              -   
Borrelia        MSH-WSD (UMLS)                                C0024198   
                MSH-WSD (UMLS)                                C0006033   
                WordNet                                  borrelia.n.01   
Cilia           MSH-WSD (UMLS)                                C0008778   
                MSH-WSD (UMLS)                                C0015422   
                WordNet                                    cilium.n.01   
                WordNet                                   eyelash.n.01   
Follicles       MSH-WSD (UMLS)                                C0221971  

**The CUI → WordNet mapping**

Comparing the two lists above, for each CUI we chose by hand the WordNet synset(s) with the same meaning. For example *ADP* `C0001459` (Adenosine Diphosphate) → `adenosine_diphosphate.n.01`. The table below shows our mapping (the dictionary `MAPPING` is in `wsd_utils.py`, because Tasks 4 and 6 use it too).

Rules we followed:
- if WordNet has **no equivalent** sense, the CUI gets an empty list `[]` (e.g. *Coronary Artery Disease*): Lesk can never be right on those abstracts;
- each WordNet synset is assigned to **at most one CUI**. Otherwise, a word like *Follicles* (one WordNet sense, two MSH-WSD senses) would make Lesk "always right" without disambiguating anything.

Some choices are subjective and are listed as limitations at the end (e.g. *Follicles*, *Respiration*, *Nurse*).

In [4]:
def mapping_table(words):
    """Our manual mapping as a table: word, CUI, UMLS name and WordNet synsets."""
    rows = []
    for word in words:
        for cui, synsets in MAPPING[word].items():
            if cui not in cui_names:                  # name not downloaded yet
                cui_names[cui] = cui_name(cui)
            rows.append({
                "word": word,
                "CUI": cui,
                "UMLS name": cui_names[cui],
                "WordNet synsets": ", ".join(synsets) if synsets else "– (not in WordNet)",
            })
    return pd.DataFrame(rows).set_index(["word", "CUI"])


mapping_table(selected_acronyms + selected_terms)

UMLS name  \
word            CUI                                                               
ADP             C0001459                                  Adenosine Diphosphate   
                C0004374                              Automatic Data Processing   
CAD             C0011905                            Computer Assisted Diagnosis   
                C1956346                                Coronary Artery Disease   
CAM             C0007578                                Cell Adhesion Molecules   
                C0178551                               chorioallantoic membrane   
CDA             C0092801                                             cladribine   
                C0002876                    Congenital dyserythropoietic anemia   
Ca              C0006675                                                calcium   
                C0006754                                             California   
                C0019564                                    Hippocampus (Brain)   
                C0006823                                                 Canada   
DDD             C0026256                                               mitotane   
                C0011037                         Dichlorodiphenyldichloroethane   
DON             C0028652                                    Nurse Administrator   
                C0012020                                     Diazooxonorleucine   
SARS            C1175175                      Severe Acute Respiratory Syndrome   
                C1175743  Severe acute respiratory syndrome-related coronavirus   
Staph           C0038160                              Staphylococcal Infections   
                C0038170                                   Genus staphylococcus   
TNT             C0077404                                             Troponin T   
                C0041070                                        Trinitrotoluene   
B-Cell Leukemia C2004493                                       Leukemia, B-Cell   
                C0023434                           Chronic Lymphocytic Leukemia   
Borrelia        C0024198                                           Lyme Disease   
                C0006033                                               Borrelia   
Cilia           C0008778                                                  Cilia   
                C0015422                                                Eyelash   
Follicles       C0221971                                Hair follicle structure   
                C0018120                                       Ovarian Follicle   
Ganglion        C0017067                                                Ganglia   
                C1258666                                            Myxoid cyst   
Nurse           C0028661                                                 Nurses   
                C0006147                                         Breast Feeding   
Pneumocystis    C0032305                                                    NaN   
                C0597258                                     Genus Pneumocystis   
Respiration     C0035203                                            Respiration   
                C0282636                                       Cell Respiration   
Sodium          C0037473                                                 sodium   
                C0037570                                        Sodium, Dietary   
Synapsis        C0039062                                               Synapses   
                C0598501                                     Chromosome Pairing   

                                                 WordNet synsets  
word            CUI                                               
ADP             C0001459              adenosine_diphosphate.n.01  
                C0004374          automatic_data_processing.n.01  
CAD             C0011905                      – (not in WordNet)  
                C1956346                      – (not in WordNet)  
CAM            

**Observations**
- Most acronyms have no sense in WordNet (*CAD*, *CAM*, *CDA*, *DDD*, *DON*: "– (not in WordNet)" for both CUIs), while most terms have at least one. So we already expect Lesk to perform much worse on acronyms.

For every abstract of the 20 words:
1. Lesk picks a WordNet sense, using the whole abstract as context (lowercased and split into words);
2. we translate the correct answer with our mapping (CUI → synsets);
3. the answer is correct if Lesk's synset is in the translated list.

We compute four numbers per word and then average them over the 10 words of each group (macro average):
- **accuracy**: correct answers / all abstracts;
- **coverage**: share of abstracts whose correct meaning exists in WordNet (the best Lesk could ever do);
- **accuracy on covered senses**: accuracy computed only on those abstracts;
- **random baseline**: expected accuracy of choosing a WordNet sense at random (e.g. 1 correct sense out of
  2 → 0.5). If Lesk is not better than this, it is not really "understanding" anything.

We also look separately at the words with **at least 2 WordNet senses**, the only ones where Lesk really has
to choose.

In [5]:
def tokenize(text):
    """Split a text into lowercase words (letters and digits only)."""
    return re.findall(r"\w+", text.lower())


def evaluate(words, mapping, acronym_list):
    """Run Lesk on every sentence of the given words and compute the metrics."""
    results = []
    for word in words:
        senses = wordnet_senses(word)
        sense_names = [s.name() for s in senses]
        instances = load_instances(word)

        n_correct = n_covered = n_correct_covered = 0
        baseline_sum = 0.0   # expected correct answers of a random choice

        for inst in instances:
            gold = mapping[word][inst["cui"]]   # WordNet translation of the correct answer

            if senses:
                answer = lesk(tokenize(inst["context"]), word.lower(), synsets=senses)
                answer = answer.name() if answer else None
            else:
                answer = None

            is_correct = answer in gold
            n_correct += is_correct
            if gold:
                n_covered += 1
                n_correct_covered += is_correct
                # random choice: probability of picking one of the correct synsets
                baseline_sum += len(set(gold) & set(sense_names)) / len(sense_names)

        results.append({
            "word": word,
            "group": "acronym" if word in acronym_list else "term",
            "n_sentences": len(instances),
            "n_wn_senses": len(senses),
            "accuracy": n_correct / len(instances),
            "coverage": n_covered / len(instances),
            "acc_covered": n_correct_covered / n_covered if n_covered else None,
            "baseline_covered": baseline_sum / n_covered if n_covered else None,
        })
    return results


def report(results):
    """Two tables: the results of each word, and the averages of each group (acronyms / terms)."""
    per_word = pd.DataFrame(results).set_index(["group", "word"]).round(3)

    summary = []
    for group in ["acronym", "term"]:
        rows = [r for r in results if r["group"] == group]
        real = [r for r in rows if r["n_wn_senses"] >= 2 and r["acc_covered"] is not None]   # Lesk really has to choose
        summary.append({
            "group": group,
            "accuracy": mean(r["accuracy"] for r in rows),
            "coverage": mean(r["coverage"] for r in rows),
            "acc. on covered senses": mean(r["acc_covered"] for r in rows),
            "real choice: Lesk": mean(r["acc_covered"] for r in real),
            "real choice: random": mean(r["baseline_covered"] for r in real),
            "real choice: words": ", ".join(r["word"] for r in real),
        })
    return per_word, pd.DataFrame(summary).set_index("group").round(3)


results = evaluate(selected_acronyms + selected_terms, MAPPING, selected_acronyms)
per_word, summary = report(results)
display(per_word)
display(summary)

n_sentences  n_wn_senses  accuracy  coverage  \
group   word                                                            
acronym ADP                      149            2     0.718     1.000   
        CAD                      198            2     0.000     0.000   
        CAM                      198            2     0.000     0.000   
        CDA                      198            0     0.000     0.000   
        Ca                       396            2     0.258     0.500   
        DDD                      198            0     0.000     0.000   
        DON                      126            7     0.000     0.000   
        SARS                     198            1     0.500     0.500   
        Staph                    198            1     0.500     0.500   
        TNT                      198            1     0.500     0.500   
term    B-Cell Leukemia          158            0     0.000     0.000   
        Borrelia                 198            1     0.500     0.500   
        Cilia                    156            2     0.628     1.000   
        Follicles                198            1     0.500     0.500   
        Ganglion                 198            1     0.500     0.500   
        Nurse                    198            7     0.470     1.000   
        Pneumocystis             198            0     0.000     0.000   
        Respiration              198            3     0.505     1.000   
        Sodium                   197            1     0.497     0.497   
        Synapsis                 134            1     0.739     0.739   

                         acc_covered  baseline_covered  
group   word                                            
acronym ADP                    0.718             0.500  
        CAD                      NaN               NaN  
        CAM                      NaN               NaN  
        CDA                      NaN               NaN  
        Ca                     0.515             0.500  
        DDD                      NaN               NaN  
        DON                      NaN               NaN  
        SARS                   1.000             1.000  
        Staph                  1.000             1.000  
        TNT                    1.000             1.000  
term    B-Cell Leukemia          NaN               NaN  
        Borrelia               1.000             1.000  
        Cilia                  0.628             0.500  
        Follicles              1.000             1.000  
        Ganglion               1.000             1.000  
        Nurse                  0.470             0.214  
        Pneumocystis             NaN               NaN  
        Respiration            0.505             0.500  
        Sodium                 1.000             1.000  
        Synapsis               1.000             1.000

,accuracy,coverage,acc. on covered senses,real choice: Lesk,real choice: random,real choice: words
group,,,,,,
acronym,0.248,0.300,0.847,0.617,0.500,"ADP, Ca"
term,0.434,0.574,0.825,0.534,0.405,"Cilia, Nurse, Respiration"


- Lesk is much worse on acronyms than on terms. The main reason is coverage, since WordNet knows the correct medical meaning in only 30% of the acronym abstracts.
- The accuracy on covered senses looks high but is misleading. Many words have only one WordNet sense, so Lesk has no choice and is "right" without really disambiguating.
- When Lesk really has to choose (ADP, Ca, Cilia, Nurse, Respiration), it is only a bit better than random: 0.617 vs 0.500 on acronyms and 0.534 vs 0.405 on terms.

In the first selection many words had zero or one WordNet sense, so the results mix two different problems:
WordNet **not knowing** the medical meaning, and Lesk **choosing badly**. To separate them, we draw a second
random selection (`random.seed(7)`), this time only among the words with **at least 2 WordNet senses**.
The main selection remains the random one required by the task; this is an additional experiment.

The table below shows, for each new word, its UMLS names and WordNet senses, as for the first selection.

In [6]:
wn_acronyms = [w for w in acronyms if len(wordnet_senses(w)) >= 2]
wn_terms    = [w for w in terms    if len(wordnet_senses(w)) >= 2]
print("Acronyms with >=2 WordNet senses:", len(wn_acronyms))
print("Terms    with >=2 WordNet senses:", len(wn_terms))

random.seed(7)
ext_acronyms = sorted(random.sample(wn_acronyms, min(10, len(wn_acronyms))))
ext_terms    = sorted(random.sample(wn_terms,    min(10, len(wn_terms))))
print("\nExtension acronyms:", ext_acronyms)
print("Extension terms   :", ext_terms)

senses_table(ext_acronyms + ext_terms) 

Acronyms with >=2 WordNet senses: 35
Terms    with >=2 WordNet senses: 49

Extension acronyms: ['ALS', 'ANA', 'CI', 'CLS', 'FAS', 'GAG', 'MRS', 'PVC', 'RA', 'US']
Extension terms   : ['Callus', 'Cell', 'Cement', 'Cilia', 'ERUPTION', 'Exercises', 'Medullary', 'Milk', 'Phosphorus', 'Ventricles']


id  \
word       source                                                   
ALS        MSH-WSD (UMLS)                                C0003372   
           MSH-WSD (UMLS)                                C0002736   
           WordNet             amyotrophic_lateral_sclerosis.n.01   
           WordNet                                  aluminum.n.01   
           WordNet                                   alabama.n.01   
ANA        MSH-WSD (UMLS)                                C0002463   
           MSH-WSD (UMLS)                                C0003243   
           WordNet                                       ana.n.01   
           WordNet                                       ana.n.02   
CI         MSH-WSD (UMLS)                                C0022326   
           MSH-WSD (UMLS)                                C0008107   
           WordNet                                     curie.n.01   
           WordNet                           hundred_and_one.s.01   
CLS        MSH-WSD (UMLS)                                C0265252   
           MSH-WSD (UMLS)                                C0343084   
           WordNet                                  chlorine.n.01   
           WordNet                                centiliter.n.01   
FAS        MSH-WSD (UMLS)                                C0015683   
           MSH-WSD (UMLS)                                C0015923   
           WordNet                    fetal_alcohol_syndrome.n.01   
           WordNet                                        fa.n.01   
GAG        MSH-WSD (UMLS)                                C0017346   
           MSH-WSD (UMLS)                                C0017973   
           WordNet                                      joke.n.01   
           WordNet                                       gag.n.02   
           WordNet                                       gag.v.01   
           WordNet                                     choke.v.02   
           WordNet                                       gag.v.03   
           WordNet                                       gag.v.04   
           WordNet                                       gag.v.05   
           WordNet                                       gag.v.06   
           WordNet                                       gag.v.07   
MRS        MSH-WSD (UMLS)                                C0025235   
           MSH-WSD (UMLS)                                C0024487   
           WordNet                                       mrs.n.01   
           WordNet                                    mister.n.01   
PVC        MSH-WSD (UMLS)                                C0151636   
           MSH-WSD (UMLS)                                C0032624   
           WordNet                        polyvinyl_chloride.n.01   
           WordNet         premature_ventricular_contraction.n.01   
RA         MSH-WSD (UMLS)                                C0002893   
           MSH-WSD (UMLS)                                C0034625   
           MSH-WSD (UMLS)                                C0003873   
           WordNet                                    radium.n.01   
           WordNet                                        ra.n.02   
           WordNet                           right_ascension.n.01   
US         MSH-WSD (UMLS)                                C0041703   
           MSH-WSD (UMLS)                                C0041618   
           WordNet                             united_states.n.01   
           WordNet                                    uracil.n.01   
           WordNet                                   uranium.n.01   
           WordNet                                         u.n.03   
Callus     MSH-WSD (UMLS)                                C0006767   
           MSH-WSD (UMLS)                                C0376154   
           WordNet                                 callosity.n.01   
           WordNet                                    callus.n.02   
           WordNet                                    callus.

The table below shows the mapping of the new words (same rules as before), then we reuse the same `evaluate` function. Subjective choices here: *Milk* (generic milk vs human milk), *Exercises* (physiotherapy vs physical exercise), *Cement* (dental cement also mapped to "adhesive material"). *Cilia* was drawn in both selections, which is expected since the two draws are independent.

In [7]:
# Manual mapping for the extension selection (same rules as MAPPING)
mapping_table(ext_acronyms + ext_terms)

UMLS name  \
word       CUI                                            
ALS        C0003372                Antilymphocyte Serum   
           C0002736       Amyotrophic Lateral Sclerosis   
ANA        C0002463        American Nurses' Association   
           C0003243             Antibodies, Antinuclear   
CI         C0022326                         Ivory Coast   
           C0008107                               Chile   
CLS        C0265252               Coffin-Lowry syndrome   
           C0343084             Capillary Leak Syndrome   
FAS        C0015683                 Fatty-acid synthase   
           C0015923              Fetal Alcohol Syndrome   
GAG        C0017346                            GAG Gene   
           C0017973                  Glycosaminoglycans   
MRS        C0025235       Melkersson-Rosenthal Syndrome   
           C0024487     Magnetic Resonance Spectroscopy   
PVC        C0151636  Premature ventricular contractions   
           C0032624                  Polyvinyl Chloride   
RA         C0002893                  Refractory anemias   
           C0034625                              radium   
           C0003873                Rheumatoid Arthritis   
US         C0041703                       United States   
           C0041618                     Ultrasonography   
Callus     C0006767                         Bone callus   
           C0376154                         Skin callus   
Cell       C0007634                               Cells   
           C1136359                      Cellular Phone   
Cement     C1706094              Dental Cement Material   
           C0011343                     Dental Cementum   
Cilia      C0008778                               Cilia   
           C0015422                             Eyelash   
ERUPTION   C0015230                           Exanthema   
           C1533692             Tooth eruption function   
Exercises  C0452240          Physical therapy exercises   
           C0015259                            Exercise   
Medullary  C0001629                     Adrenal Medulla   
           C0025148                   Medulla Oblongata   
Milk       C0026131               Milk (body substance)   
           C0026140                         Milk, Human   
Phosphorus C0080014                  Dietary Phosphorus   
           C0031705                          phosphorus   
Ventricles C0018827                     Heart Ventricle   
           C0007799                 Cerebral Ventricles   

                                            WordNet synsets  
word       CUI                                               
ALS        C0003372                      – (not in WordNet)  
           C0002736      amyotrophic_lateral_sclerosis.n.01  
ANA        C0002463                      – (not in WordNet)  
           C0003243                      – (not in WordNet)  
CI         C0022326                      – (not in WordNet)  
           C0008107                      – (not in WordNet)  
CLS        C0265252                      – (not in WordNet)  
           C0343084                      – (not in WordNet)  
FAS        C0015683                      – (not in WordNet)  
           C0015923             fetal_alcohol_syndrome.n.01  
GAG        C0017346                      – (not in WordNet)  
           C0017973                      – (not in WordNet)  
MRS        C0025235                      – (not in WordNet)  
           C0024487                      – (not in WordNet)  
PVC        C0151636  premature_ventricular_contraction.n.01  
           C0032624                 polyvinyl_chloride.n.01  
RA         C0002893                      – (not in WordNet)  
           C0034625                             radium.n.01  
           C0003873                      – (not in WordNet)  
US         C0041703                      united_states.n.01  
           C0041618                      – (not in WordNet)  
Callus     C0006767                             callus.n.02  
           C0376154 

In [8]:
ext_results = evaluate(ext_acronyms + ext_terms, MAPPING, ext_acronyms)
ext_per_word, ext_summary = report(ext_results)
display(ext_per_word)
display(ext_summary)

n_sentences  n_wn_senses  accuracy  coverage  acc_covered  \
group   word                                                                    
acronym ALS                 198            3     0.495     0.500        0.990   
        ANA                 198            2     0.000     0.000          NaN   
        CI                  183            2     0.000     0.000          NaN   
        CLS                  34            2     0.000     0.000          NaN   
        FAS                 198            2     0.500     0.500        1.000   
        GAG                 198            9     0.000     0.000          NaN   
        MRS                 166            2     0.000     0.000          NaN   
        PVC                 198            2     0.813     1.000        0.813   
        RA                  297            3     0.000     0.333        0.000   
        US                  198            4     0.116     0.500        0.232   
term    Callus              150            5     0.567     1.000        0.567   
        Cell                198            7     0.439     1.000        0.439   
        Cement              185            8     0.049     1.000        0.049   
        Cilia               156            2     0.628     1.000        0.628   
        ERUPTION            198            6     0.242     1.000        0.242   
        Exercises           198           10     0.369     1.000        0.369   
        Medullary           198            3     0.515     1.000        0.515   
        Milk                198            7     0.040     1.000        0.040   
        Phosphorus          198            2     0.500     0.500        1.000   
        Ventricles          198            2     0.677     1.000        0.677   

                    baseline_covered  
group   word                          
acronym ALS                    0.333  
        ANA                      NaN  
        CI                       NaN  
        CLS                      NaN  
        FAS                    0.500  
        GAG                      NaN  
        MRS                      NaN  
        PVC                    0.500  
        RA                     0.333  
        US                     0.250  
term    Callus                 0.200  
        Cell                   0.143  
        Cement                 0.192  
        Cilia                  0.500  
        ERUPTION               0.167  
        Exercises              0.100  
        Medullary              0.333  
        Milk                   0.143  
        Phosphorus             0.500  
        Ventricles             0.500

,accuracy,coverage,acc. on covered senses,real choice: Lesk,real choice: random,real choice: words
group,,,,,,
acronym,0.192,0.283,0.607,0.607,0.383,"ALS, FAS, PVC, RA, US"
term,0.403,0.950,0.453,0.453,0.278,"Callus, Cell, Cement, Cilia, ERUPTION, Exercises, Medullary, Milk, Phosphorus, Ventricles"


**Observations**
- **Acronyms → a WordNet problem.** Even choosing only acronyms that exist in WordNet, coverage stays at 28%: WordNet has *MRS* ("a form of address for a married woman"), *GAG* ("a humorous remark") or *CI* (the curie and the number 101), but none of their medical meanings.
- **Terms → an algorithm problem.** Coverage is almost complete (95%), yet Lesk is right on only 40% of the abstracts (45% of the covered ones): even with all the information available, it chooses wrong more than half of the time.
- Some words are worse than random: *Cement* 0.049 vs 0.192, *Milk* 0.040 vs 0.143, *RA* 0.000 vs 0.333. Being worse than chance means that Lesk does not fail randomly: it **always picks the same wrong sense**. We check this in the next step.
- Terms are better than acronyms also in this selection, which makes the conclusion more reliable despite the small samples.

For the words that performed worse than random, we count how many times Lesk returns each sense.
`Counter` adds 1 to an answer every time Lesk gives it, and `most_common()` sorts them by frequency.

In [9]:
# let's see which senses Lesk picks most often (for error analysis)
for word in ["Cement", "Milk", "RA", "US"]:
    senses = wordnet_senses(word)
    answers = Counter()
    for inst in load_instances(word):
        a = lesk(tokenize(inst["context"]), word.lower(), synsets=senses)
        answers[a.name() if a else None] += 1
    print(f"\n=== {word}")
    for sense, count in answers.most_common():
        definition = wn.synset(sense).definition() if sense else ""
        print(f"   {count:4}  {sense:28} {definition[:70]}")


=== Cement
    160  cement.n.02                  a building material that is a powder made of a mixture of calcined lim
     10  cement.n.04                  any of various materials used by dentists to fill cavities in teeth
      8  cement.n.01                  concrete pavement is sometimes referred to as cement
      7  cementum.n.01                a specialized bony substance covering the root of a tooth

=== Milk
    174  milk.n.03                    a river that rises in the Rockies in northwestern Montana and flows ea
     18  milk.n.01                    a white nutritious liquid secreted by mammals and used as food by huma
      3  milk.v.01                    take milk from female mammals
      2  milk.n.02                    produced by mammary glands of female mammals for feeding their young
      1  milk.v.03                    add milk to

=== RA
    288  right_ascension.n.01         (astronomy) the equatorial coordinate specifying the angle, measured e
      8  ra.n.02

**The errors are systematic**: for *Milk* Lesk returns "a river in Montana" in 174/198 abstracts, for *RA*
"right ascension (astronomy)" in 288/297, for *US* "uracil" in 154/198, and for *Cement* "a building
material" in 160/185.

We can guess the cause of this by looking for example at the winning definition of *Milk*: "a river that rises **in the** Rockies **in** northwestern Montana **and** flows eastward **to** become…". It is long and full of **stopwords**
(*the, in, and, to, of*), which appear in every abstract. Lesk counts them as if they were meaningful, so the
definition with the most "empty words" wins. This is the same problem we saw in Task 1, where Lesk chose a
sense because of the word "to".

The error analysis suggests two simple improvements, which we compare in four versions:

- **Stopword removal**: we remove the English stopwords (NLTK list) from the context. Since Lesk counts the
  words in common, a stopword missing from the context can no longer count.
- **POS filtering**: we give Lesk only the **noun** senses (`pos='n'`), discarding verbs such as
  *milk.v.01* ("take milk from female mammals"). If a word has no noun senses (e.g. *Medullary*, an
  adjective) we keep all its senses. Side effect: for *Nurse*, breastfeeding was mapped to a verb
  (`breastfeed.v.01`), so with POS filtering Lesk can no longer get it right.


In [10]:
def candidate_senses(word, use_pos):
    """WordNet senses of the word; with use_pos only nouns (if the word has any)."""
    senses = wordnet_senses(word)
    if use_pos:
        nouns = [s for s in senses if s.pos() == "n"]
        if nouns:                 # e.g. "Medullary" has only adjectives: keep all senses
            senses = nouns
    return senses

def evaluate_variant(words, mapping, use_pos, remove_stopwords):
    """Accuracy and accuracy on covered senses of one Lesk variant, for each word."""
    scores = {}
    for word in words:
        senses = candidate_senses(word, use_pos)
        instances = load_instances(word)
        n_correct = n_covered = n_correct_covered = 0
        for inst in instances:
            gold = mapping[word][inst["cui"]]
            context = tokenize(inst["context"])
            if remove_stopwords:
                context = [t for t in context if t not in STOPWORDS]
            if senses:
                answer = lesk(context, word.lower(), synsets=senses)
                answer = answer.name() if answer else None
            else:
                answer = None
            is_correct = answer in gold
            n_correct += is_correct
            if gold:
                n_covered += 1
                n_correct_covered += is_correct
        scores[word] = (n_correct / len(instances),
                        n_correct_covered / n_covered if n_covered else None)
    return scores


VARIANTS = {
    "Lesk (base)":             dict(use_pos=False, remove_stopwords=False),
    "Lesk + POS":              dict(use_pos=True,  remove_stopwords=False),
    "Lesk + stopwords":        dict(use_pos=False, remove_stopwords=True),
    "Lesk + stopwords + POS":  dict(use_pos=True,  remove_stopwords=True),
}

SELECTIONS = {
    "Selection 1 (random)":       (selected_acronyms, selected_terms, MAPPING),
    "Selection 2 (>=2 WN senses)": (ext_acronyms, ext_terms, MAPPING),
}

all_scores = {}   # saved for the error analysis below
table = []
for sel_name, (acr, ter, mapping) in SELECTIONS.items():
    for var_name, params in VARIANTS.items():
        scores = evaluate_variant(acr + ter, mapping, **params)
        all_scores[(sel_name, var_name)] = scores
        table.append({"selection": sel_name, "variant": var_name,
                      "acronyms acc": mean(scores[w][0] for w in acr),
                      "acronyms cov": mean(scores[w][1] for w in acr),
                      "terms acc": mean(scores[w][0] for w in ter),
                      "terms cov": mean(scores[w][1] for w in ter)})

pd.DataFrame(table).set_index(["selection", "variant"]).round(3)

acronyms acc  \
selection                   variant                                
Selection 1 (random)        Lesk (base)                    0.248   
                            Lesk + POS                     0.248   
                            Lesk + stopwords               0.245   
                            Lesk + stopwords + POS         0.245   
Selection 2 (>=2 WN senses) Lesk (base)                    0.192   
                            Lesk + POS                     0.192   
                            Lesk + stopwords               0.256   
                            Lesk + stopwords + POS         0.256   

                                                    acronyms cov  terms acc  \
selection                   variant                                           
Selection 1 (random)        Lesk (base)                    0.847      0.434   
                            Lesk + POS                     0.847      0.436   
                            Lesk + stopwords               0.842      0.433   
                            Lesk + stopwords + POS         0.842      0.449   
Selection 2 (>=2 WN senses) Lesk (base)                    0.607      0.403   
                            Lesk + POS                     0.607      0.408   
                            Lesk + stopwords               0.881      0.425   
                            Lesk + stopwords + POS         0.881      0.478   

                                                    terms cov  
selection                   variant                            
Selection 1 (random)        Lesk (base)                 0.825  
                            Lesk + POS                  0.829  
                            Lesk + stopwords            0.824  
                            Lesk + stopwords + POS      0.845  
Selection 2 (>=2 WN senses) Lesk (base)                 0.453  
                            Lesk + POS                  0.458  
                            Lesk + stopwords            0.475  
                            Lesk + stopwords + POS      0.528

**Observations**
- **Selection 1**: almost no change, because most words have only one WordNet sense, so there is nothing to improve.
- **Selection 2**: removing stopwords is the most useful change, especially on acronyms (0.607 → 0.881 on covered senses).
- POS alone helps very little, because most wrong senses are nouns too (e.g. the river *Milk*).
- Stopwords + POS together is the best version on terms (0.453 → 0.528).

After removing stopwords, some abstracts may share **no word at all** with any definition. In that case
Lesk has no clue and simply returns the **first WordNet sense** (because of how ties are handled, see the note
at the top). So part of the improvement could come from this fallback rather than from real word overlap.

For each word we compute the **zero-overlap rate**: the share of abstracts in which no definition shares any
word with the context.

In [11]:
def zero_overlap_rate(word, use_pos, remove_stopwords):
    """Fraction of sentences where NO sense shares a word with the context
    (in that case Lesk simply returns the first WordNet sense)."""
    senses = candidate_senses(word, use_pos)
    if not senses:
        return None
    zero = 0
    instances = load_instances(word)
    for inst in instances:
        context = set(tokenize(inst["context"]))
        if remove_stopwords:
            context -= STOPWORDS
        best = max(len(context & set(s.definition().split())) for s in senses)
        zero += (best == 0)
    return zero / len(instances)


sel = "Selection 2 (>=2 WN senses)"
base = all_scores[(sel, "Lesk (base)")]
best = all_scores[(sel, "Lesk + stopwords + POS")]

print(f"{'word':12} {'base':>6} {'best':>6} {'change':>7} {'zero_overlap':>13}")
for word in ext_acronyms + ext_terms:
    zr = zero_overlap_rate(word, use_pos=True, remove_stopwords=True)
    zr = f"{zr:.2f}" if zr is not None else "-"
    print(f"{word:12} {base[word][0]:6.3f} {best[word][0]:6.3f} "
          f"{best[word][0] - base[word][0]:+7.3f} {zr:>13}")

word           base   best  change  zero_overlap
ALS           0.495  0.500  +0.005          0.18
ANA           0.000  0.000  +0.000          0.96
CI            0.000  0.000  +0.000          0.64
CLS           0.000  0.000  +0.000          0.41
FAS           0.500  0.500  +0.000          0.35
GAG           0.000  0.000  +0.000          0.97
MRS           0.000  0.000  +0.000          0.84
PVC           0.813  0.919  +0.106          0.05
RA            0.000  0.195  +0.195          0.52
US            0.116  0.449  +0.333          0.53
Callus        0.567  0.880  +0.313          0.20
Cell          0.439  0.389  -0.051          0.28
Cement        0.049  0.195  +0.146          0.00
Cilia         0.628  0.641  +0.013          0.74
ERUPTION      0.242  0.298  +0.056          0.55
Exercises     0.369  0.384  +0.015          0.16
Medullary     0.515  0.308  -0.207          0.39
Milk          0.040  0.455  +0.414          0.25
Phosphorus    0.500  0.500  +0.000          0.47
Ventricles    0.677 

**Observations**
- **Real improvements**: *Milk* (+41), *Callus* (+31), *Cement* (+15) have a low zero-overlap rate, so Lesk now finds useful words in common (e.g. *dental*, *teeth*).
- **Improvements by luck**: for *US* and *RA* about half of the abstracts have no overlap at all, so Lesk just returns the first WordNet sense, which happens to be correct. This is the "back-off to the most frequent sense" described by Vasilescu et al. (2004).
- **The same fallback can hurt**: for *Medullary* the first sense ("bone marrow") is wrong, so accuracy drops from 0.515 to 0.308.

Other ideas to improve Lesk algorithm: 
- **Most Frequent Sense (MFS) baseline**: always choose the most common sense. In WordNet,
  senses are listed by frequency, so we simply take the **first sense**. This tells us whether Lesk is
  better than this simple rule, and it is also what Lesk falls back on when there is zero overlap.
- **Richer signature**: the slides define the signature of a sense as its **gloss + examples**,
  but `nltk.wsd.lesk` only uses the gloss. We write our own simplified Lesk that also uses the WordNet
  **examples** and **synonyms** (lemma names) of each sense.

The target word itself is removed from the signature: it appears in every abstract, so it would give a
free point to every sense that contains it and penalise senses with a different name (e.g. *eyelash* for
*Cilia*). All methods use stopword removal and POS filtering, so they can be compared with the best variant
found above.

In [12]:
def build_signature(sense, target, use_examples):
    """Set of words describing a sense: gloss (+ examples and synonyms if use_examples)."""
    text = sense.definition()
    if use_examples:
        text += " " + " ".join(sense.examples())                        # example sentences
        text += " " + " ".join(l.replace("_", " ") for l in sense.lemma_names())  # synonyms
    words = set(tokenize(text)) - STOPWORDS
    words -= set(tokenize(target))          # remove the target word itself
    return words


def my_lesk(context, target, senses, use_examples):
    """Simplified Lesk: choose the sense whose signature shares most words with the context."""
    context = set(context)
    return max(senses, key=lambda s: len(context & build_signature(s, target, use_examples)))


# Each method receives (word, context tokens without stopwords, candidate senses) and returns a synset
METHODS = {
    "MFS (first WordNet sense)":
        lambda word, ctx, senses: senses[0],
    "NLTK Lesk (stopwords + POS)":
        lambda word, ctx, senses: lesk(ctx, word.lower(), synsets=senses),
    "Our Lesk: gloss only":
        lambda word, ctx, senses: my_lesk(ctx, word, senses, use_examples=False),
    "Our Lesk: gloss + examples + synonyms":
        lambda word, ctx, senses: my_lesk(ctx, word, senses, use_examples=True),
}


def evaluate_method(words, mapping, method):
    """Accuracy and accuracy on covered senses of one method, for each word."""
    scores = {}
    for word in words:
        senses = candidate_senses(word, use_pos=True)
        instances = load_instances(word)
        n_correct = n_covered = n_correct_covered = 0
        for inst in instances:
            gold = mapping[word][inst["cui"]]
            context = [t for t in tokenize(inst["context"]) if t not in STOPWORDS]
            answer = method(word, context, senses).name() if senses else None
            is_correct = answer in gold
            n_correct += is_correct
            if gold:
                n_covered += 1
                n_correct_covered += is_correct
        scores[word] = (n_correct / len(instances),
                        n_correct_covered / n_covered if n_covered else None)
    return scores


table = []
for sel_name, (acr, ter, mapping) in SELECTIONS.items():
    for name, method in METHODS.items():
        scores = evaluate_method(acr + ter, mapping, method)
        table.append({"selection": sel_name, "method": name,
                      "acronyms acc": mean(scores[w][0] for w in acr),
                      "acronyms cov": mean(scores[w][1] for w in acr),
                      "terms acc": mean(scores[w][0] for w in ter),
                      "terms cov": mean(scores[w][1] for w in ter)})

pd.DataFrame(table).set_index(["selection", "method"]).round(3)

acronyms acc  \
selection                   method                                                
Selection 1 (random)        MFS (first WordNet sense)                     0.241   
                            NLTK Lesk (stopwords + POS)                   0.245   
                            Our Lesk: gloss only                          0.247   
                            Our Lesk: gloss + examples + synonyms         0.254   
Selection 2 (>=2 WN senses) MFS (first WordNet sense)                     0.233   
                            NLTK Lesk (stopwords + POS)                   0.256   
                            Our Lesk: gloss only                          0.257   
                            Our Lesk: gloss + examples + synonyms         0.268   

                                                                   acronyms cov  \
selection                   method                                                
Selection 1 (random)        MFS (first WordNet sense)                     0.833   
                            NLTK Lesk (stopwords + POS)                   0.842   
                            Our Lesk: gloss only                          0.847   
                            Our Lesk: gloss + examples + synonyms         0.876   
Selection 2 (>=2 WN senses) MFS (first WordNet sense)                     0.900   
                            NLTK Lesk (stopwords + POS)                   0.881   
                            Our Lesk: gloss only                          0.881   
                            Our Lesk: gloss + examples + synonyms         0.918   

                                                                   terms acc  \
selection                   method                                             
Selection 1 (random)        MFS (first WordNet sense)                  0.437   
                            NLTK Lesk (stopwords + POS)                0.449   
                            Our Lesk: gloss only                       0.449   
                            Our Lesk: gloss + examples + synonyms      0.456   
Selection 2 (>=2 WN senses) MFS (first WordNet sense)                  0.297   
                            NLTK Lesk (stopwords + POS)                0.478   
                            Our Lesk: gloss only                       0.493   
                            Our Lesk: gloss + examples + synonyms      0.508   

                                                                   terms cov  
selection                   method                                            
Selection 1 (random)        MFS (first WordNet sense)                  0.829  
                            NLTK Lesk (stopwords + POS)                0.845  
                            Our Lesk: gloss only                       0.845  
                            Our Lesk: gloss + examples + synonyms      0.854  
Selection 2 (>=2 WN senses) MFS (first WordNet sense)                  0.347  
                            NLTK Lesk (stopwords + POS)                0.528  
                            Our Lesk: gloss only                       0.543  
                            Our Lesk: gloss + examples + synonyms      0.558

**Observations**
- **Selection 1**: all methods give almost the same results, because most words have only one WordNet sense.
- **Lesk vs MFS on terms in Selection 2**: Lesk is clearly better than always choosing the first sense
  (0.558 vs 0.347 on covered senses). Here the context really helps.
- **Lesk vs MFS on acronyms**: MFS is as good as Lesk (0.900 vs 0.881). For acronyms such as *US*, *RA*,
  *ALS* the first WordNet sense happens to be the one used in medical texts, so the context adds little.
- Cleaning the gloss helps a bit. Our Lesk with the gloss only is slightly better than NLTK
  (0.543 vs 0.528 on terms), because we also remove punctuation and stopwords from the definition.
- **Examples and synonyms help further**, as suggested by the slides: this is the best method on both groups (acronyms 0.918, terms 0.558 on covered senses). The gain is small because WordNet examples are short and often missing.

## Summary
- **Main result**: standard Lesk reaches **24.8% on acronyms** and **43.4% on terms**.
- **Acronyms fail because of WordNet**: it is a general dictionary and does not contain most medical
  abbreviations (coverage ~30%) (See Burgun & Bodenreider (2001))
- **Terms fail because of the algorithm**: even with 95% coverage, standard Lesk is right only ~45% of the time.
- **Many errors are systematic**: long definitions full of stopwords always win (*Milk* → a river, *RA* → astronomy).
- **Improvements**: removing stopwords, filtering by POS and adding WordNet examples and synonyms to the
  signature raise the accuracy on covered senses from 0.607 to 0.918 on acronyms and from 0.453 to 0.558 on terms.
- On **terms** the improved Lesk is clearly better than always choosing the most frequent sense,  while
  on **acronyms** it is only slightly better, because most of the gain comes from falling back
  on the first WordNet sense.

**Limitations**: the CUI → WordNet mapping was done by hand (some choices are subjective, e.g. *Follicles*,
*Milk*), and 20 words per selection is a small sample.